#### Helix RAG Demo

This notebook demonstrates a full RAG (Retrieval-Augmented Generation) pipeline using:
- **Helix** — our custom-built vector database (running locally)
- **Google Gemini** — `gemini-embedding-2` for embeddings, `gemini-3.1-flash-lite` for generation
- **Source document** — Laxmi Narayana Pattanayak's Resume

Each cell tests one specific Helix API endpoint.

#### Prerequisites
```bash
pip install langchain-text-splitters google-genai requests python-dotenv
```
Start the server:
```bash
#### From the repo root
$env:GOARCH="amd64"
& "C:\Users\Sumit Amit\go_amd64\go\bin\go.exe" run ./cmd/helix --port 8000 --db ./RAG/rag_demo.db
```

#### Cell 0 — Setup: Imports & Configuration

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv
from google import genai

# ── Configuration ───────────────────────────────────────────────
load_dotenv()
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
assert GEMINI_API_KEY, "Set the GEMINI_API_KEY environment variable before running"

BASE_URL       = "http://localhost:8000"   # Helix server
COLLECTION     = "resume_rag"              # collection name
EMBED_MODEL    = "gemini-embedding-2"      # embedding model
GEN_MODEL      = "gemini-3.1-flash-lite"        # generation model
EMBED_DIM      = 3072                      # gemini-embedding-2 output dimension

client = genai.Client(api_key=GEMINI_API_KEY)

def embed(text: str) -> list[float]:
    """Embed a single string with gemini-embedding-2."""
    result = client.models.embed_content(
        model=EMBED_MODEL,
        contents=text
    )
    return result.embeddings[0].values

def pretty(resp: requests.Response):
    """Pretty-print a requests.Response."""
    print(f"HTTP {resp.status_code}")
    try:
        print(json.dumps(resp.json(), indent=2))
    except Exception:
        print(resp.text)

print("✅ Setup complete")

#### Cell 1 — Chunk the Resume

Split the resume into semantically meaningful sections. Each section becomes one vector in the database.

In [ ]:
# Raw resume markdown text
RESUME_MARKDOWN = """# Laxmi Narayana Pattanayak
Contact: +91-6372830696, laxminarayana3101@gmail.com.
LinkedIn: linkedin.com/in/LaxmiNarayana31. GitHub: github.com/LaxmiNarayana31.

## Education
- Master of Computer Application from GIET University, Gunupur, Odisha (CGPA 8.49, Oct 2022 – May 2024).
- Bachelor of Computer Application from Berhampur University (CGPA 8.59, Jul 2019 – Jul 2022).

## Technical Skills
- Languages: Python, JavaScript, SQL.
- AI/ML & LLMs: PyTorch, Scikit-learn, NLP, RAG, LangChain, LangGraph, LlamaIndex, Prompt Engineering, AutoGen, CrewAI, OpenAI, Claude, Gemini APIs.
- Backend & Databases: FastAPI, Flask, Node.js, MySQL, MongoDB, Redis, FAISS, Qdrant, ChromaDB.
- Tools: Git, Docker, Linux, Postman, Hugging Face, Google Colab, AWS SageMaker, Bedrock, EC2, Google Vertex AI.

## Work Experience
### AI Engineer | Cognitbotz Solutions Pvt Ltd | March 2026 – Present
- Built FastAPI pipelines for automated PDF ingestion from govt. portals, cutting data collection time by 90%.
- Developed Python/Pandas ETL pipeline processing 2K+ records, reducing manual extraction effort by 85%.
- Implemented Pydantic rule-based validation, reducing manual validation time by 80%.

### Software Engineer | Verve Systems Pvt Ltd | Oct 2024 – March 2026
- Built multi-LLM conversational AI platform for natural language querying across 5+ databases.
- Engineered Text-to-SQL system with schema-aware LLM query generation.
- Improved RAG retrieval using hybrid search (BM25 + vector), boosting precision by 40%, cutting latency by 60%, cost by 85%.
- Built BI assistant converting NL queries into dashboards.
- Orchestrated 15+ AI agents for automated planning and enterprise workflow execution.
- Developed computer vision APIs for OCR, image similarity, and object detection (85% accuracy).

### Software Engineer Intern | Verve Systems Pvt Ltd | May 2024 – Oct 2024
- Delivered secure authentication and recovery mechanisms, reducing credential reset time by 40%.
- Created RBAC APIs, elevating operational performance by 25%.
- Streamlined Excel-to-JSON conversion for 10K+ product entries, cutting manual effort by 85%.

## Projects
- DocsQuery AI — document ingestion and LLM interaction handling 300+ daily sessions, optimized FAISS indexing improving response speed by 50%, LangChain context-aware retrieval (85% accuracy).
- MovieSense — content-based recommendation system using vector similarity, 70%+ precision.
- WebChat — real-time web chat with LLMs and vector search (80% accuracy), reduced information search time by 50%.

## Publications & Certifications
- Publications: Future of AI: Types, Capabilities & Applications. Why TypeScript for Scalable Development. JavaScript Execution Explained Simply.
- Certifications: Python for Data Science, AI and Development (IBM). Career Essentials in Generative AI (Microsoft & LinkedIn).
"""

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize RecursiveCharacterTextSplitter for dynamic chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

raw_chunks = text_splitter.split_text(RESUME_MARKDOWN)

# Construct chunk payloads with IDs and metadata
RESUME_CHUNKS = []
for i, chunk_text in enumerate(raw_chunks):
    # Try to heuristically identify the section for metadata filtering demonstration
    section = "general"
    lower_text = chunk_text.lower()
    if "education" in lower_text:
        section = "education"
    elif "skills" in lower_text:
        section = "skills"
    elif "experience" in lower_text or "engineer" in lower_text or "intern" in lower_text:
        section = "experience"
    elif "project" in lower_text:
        section = "projects"
    elif "certification" in lower_text or "publication" in lower_text:
        section = "publications_certifications"

    RESUME_CHUNKS.append({
        "id": f"chunk_{i}",
        "text": chunk_text,
        "metadata": {
            "section": section,
            "chunk_index": i
        }
    })

print(f"✅ {len(RESUME_CHUNKS)} chunks prepared dynamically:")
for c in RESUME_CHUNKS:
    print(f"  [{c['id']}] (section={c['metadata']['section']}): {c['text'][:70]}...")


#### Cell 2 — Endpoint: `POST /collections`

Create a collection with Cosine similarity metric and `textField=text` for hybrid search support.

In [ ]:
resp = requests.post(f"{BASE_URL}/collections", json={
    "name":      COLLECTION,
    "dimension": EMBED_DIM,
    "metric":    0,          # 0 = Cosine
    "config":    {"textField": "text"}  # enables hybrid/BM25 search
})

print("── POST /collections ──")
pretty(resp)

#### Cell 3 — Endpoint: `GET /collections`

List all collections to confirm creation.

In [ ]:
resp = requests.get(f"{BASE_URL}/collections")

print("── GET /collections ──")
pretty(resp)

#### Cell 4 — Endpoint: `POST /collections/{name}/insert`

Insert the **profile** chunk individually as a single insert call.

In [ ]:
chunk = RESUME_CHUNKS[0]  # profile chunk
vector = embed(chunk["text"])

resp = requests.post(f"{BASE_URL}/collections/{COLLECTION}/insert", json={
    "id":       chunk["id"],
    "vector":   vector,
    "metadata": {**chunk["metadata"], "text": chunk["text"]}
})

print(f"── POST /collections/{COLLECTION}/insert (id={chunk['id']}) ──")
pretty(resp)

#### Cell 5 — Endpoint: `POST /collections/{name}/batch`

Batch-insert the remaining 7 chunks in a single API call.

In [ ]:
remaining_chunks = RESUME_CHUNKS[1:]  # all except profile (already inserted)

print(f"Embedding {len(remaining_chunks)} chunks...")
items = []
for chunk in remaining_chunks:
    vector = embed(chunk["text"])
    items.append({
        "id":       chunk["id"],
        "vector":   vector,
        "metadata": {**chunk["metadata"], "text": chunk["text"]}
    })
    print(f"  ✓ embedded [{chunk['id']}]")

resp = requests.post(f"{BASE_URL}/collections/{COLLECTION}/batch", json={"items": items})

print(f"\n── POST /collections/{COLLECTION}/batch ──")
pretty(resp)

#### Cell 6 — Endpoint: `POST /collections/{name}/upsert`

Upsert the **profile** chunk with an updated metadata field (adds `updated=true`). This tests overwriting an existing vector without error.

In [ ]:
chunk = RESUME_CHUNKS[0]  # profile chunk
vector = embed(chunk["text"])

resp = requests.post(f"{BASE_URL}/collections/{COLLECTION}/upsert", json={
    "id":       chunk["id"],
    "vector":   vector,
    "metadata": {**chunk["metadata"], "text": chunk["text"], "updated": True}
})

print(f"── POST /collections/{COLLECTION}/upsert (id={chunk['id']}) ──")
pretty(resp)

#### Cell 7 — Endpoint: `POST /collections/{name}/search`

Pure **vector search**: embed a query and find the top-3 most semantically similar resume sections.

In [ ]:
query = "What AI and LLM experience does the candidate have?"
query_vec = embed(query)

resp = requests.post(f"{BASE_URL}/collections/{COLLECTION}/search", json={
    "vector": query_vec,
    "top_k":  3,
})

print(f"── POST /collections/{COLLECTION}/search ──")
print(f"Query: \"{query}\"\n")
data = resp.json()
for i, r in enumerate(data["results"], 1):
    print(f"[{i}] id={r['id']}  score={r['score']:.4f}")
    print(f"     {r['metadata'].get('text', '')[:120]}...\n")

#### Cell 8 — Endpoint: `POST /collections/{name}/search` (with metadata filter)

Filtered vector search — restrict results to only the **experience** section.

In [ ]:
query = "Tell me about the candidate's work at Verve Systems"
query_vec = embed(query)

resp = requests.post(f"{BASE_URL}/collections/{COLLECTION}/search", json={
    "vector": query_vec,
    "top_k":  5,
    "filter": {"section": "experience"}  # only experience chunks
})

print(f"── POST /collections/{COLLECTION}/search (filter: section=experience) ──")
print(f"Query: \"{query}\"\n")
data = resp.json()
for i, r in enumerate(data["results"], 1):
    print(f"[{i}] id={r['id']}  score={r['score']:.4f}  company={r['metadata'].get('company')}")
    print(f"     {r['metadata'].get('text', '')[:120]}...\n")

#### Cell 9 — Endpoint: `POST /collections/{name}/hybrid-search`

**Hybrid search** combining BM25 keyword search + vector similarity, merged with Reciprocal Rank Fusion (RRF, k=60). This is the most powerful search mode.

In [ ]:
query = "RAG pipeline and hybrid search experience"
query_vec = embed(query)

resp = requests.post(f"{BASE_URL}/collections/{COLLECTION}/hybrid-search", json={
    "text_query": query,         # BM25 keyword signal
    "vector":     query_vec,     # semantic vector signal
    "top_k":      3,
})

print(f"── POST /collections/{COLLECTION}/hybrid-search ──")
print(f"Query: \"{query}\"\n")
data = resp.json()
for i, r in enumerate(data["results"], 1):
    print(f"[{i}] id={r['id']}  rrf_score={r['score']:.6f}")
    print(f"     {r['metadata'].get('text', '')[:120]}...\n")

#### Cell 10 — Endpoint: `POST /collections/{name}/hybrid-search` (text-only BM25 mode)

Pure BM25 mode — omit the vector, use only keyword matching.

In [ ]:
resp = requests.post(f"{BASE_URL}/collections/{COLLECTION}/hybrid-search", json={
    "text_query": "FastAPI ETL Python",  # keyword-only, no vector
    "top_k":      3,
})

print(f"── POST /collections/{COLLECTION}/hybrid-search (text-only BM25) ──")
data = resp.json()
for i, r in enumerate(data["results"], 1):
    print(f"[{i}] id={r['id']}  bm25_score={r['score']:.6f}")
    print(f"     {r['metadata'].get('text', '')[:120]}...\n")

#### Cell 11 — Endpoint: `GET /collections/{name}/health`

Check HNSW graph health — node count, tombstoned nodes, average degree per layer, and whether a rebuild is recommended.

In [ ]:
resp = requests.get(f"{BASE_URL}/collections/{COLLECTION}/health")

print(f"── GET /collections/{COLLECTION}/health ──")
pretty(resp)

#### Cell 12 — Full RAG Pipeline

End-to-end RAG: retrieve relevant resume chunks via hybrid search, then pass them as context to Gemini to generate a grounded answer.

In [ ]:
def rag_answer(question: str, top_k: int = 3) -> str:
    """Retrieve relevant resume chunks and generate a grounded answer."""
    # 1. Embed the question
    query_vec = embed(question)

    # 2. Hybrid search — best of vector + keyword
    resp = requests.post(f"{BASE_URL}/collections/{COLLECTION}/hybrid-search", json={
        "text_query": question,
        "vector":     query_vec,
        "top_k":      top_k,
    })
    results = resp.json().get("results", [])

    # 3. Build context from retrieved chunks
    context_parts = []
    for r in results:
        section = r["metadata"].get("section", "unknown")
        text    = r["metadata"].get("text", "")
        context_parts.append(f"[{section}] {text}")
    context = "\n\n".join(context_parts)

    # 4. Generate answer with Gemini
    prompt = f"""You are a helpful assistant answering questions about a candidate's resume.
Use ONLY the context below to answer. Do not make up information.

Context:
{context}

Question: {question}

Answer:"""

    gen_resp = client.models.generate_content(
        model=GEN_MODEL,
        contents=prompt
    )
    return gen_resp.text


# ── Test questions ──────────────────────────────────────────────
questions = [
    "What is the candidate's highest qualification?",
    "What AI/ML frameworks does the candidate know?",
    "Describe the candidate's experience with RAG systems.",
    "What projects has the candidate built involving vector search?",
    "Which companies has the candidate worked at and in what roles?",
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"{'─'*60}")
    answer = rag_answer(q)
    print(f"A: {answer}")

#### Cell 13 — Endpoint: `POST /collections/{name}/rebuild`

Trigger a zero-downtime HNSW graph rebuild. The buffered-replay mechanism ensures no inserts are lost during the rebuild.

In [ ]:
resp = requests.post(f"{BASE_URL}/collections/{COLLECTION}/rebuild")

print(f"── POST /collections/{COLLECTION}/rebuild ──")
pretty(resp)

#### Cell 14 — Endpoint: `DELETE /collections/{name}/vectors/{id}`

Delete a single vector by ID, then confirm it no longer appears in search results.

In [ ]:
# Find the chunk ID corresponding to publications/certifications
del_id = next((c["id"] for c in RESUME_CHUNKS if c["metadata"]["section"] == "publications_certifications"), "chunk_7")
resp = requests.delete(f"{BASE_URL}/collections/{COLLECTION}/vectors/{del_id}")
print(f"── DELETE /collections/{COLLECTION}/vectors/{del_id} ──")
print(f"HTTP {resp.status_code}")

# Confirm it's gone from search
query_vec = embed("IBM Python certification Microsoft Generative AI")
resp2 = requests.post(f"{BASE_URL}/collections/{COLLECTION}/search", json={
    "vector": query_vec,
    "top_k":  5,
})
ids_returned = [r["id"] for r in resp2.json().get("results", [])]
print(f"\nIDs in search results after delete: {ids_returned}")
assert del_id not in ids_returned, f"{del_id} should not appear after deletion!"
print(f"✅ '{del_id}' correctly absent from results")


#### Cell 15 — Endpoint: `DELETE /collections/{name}`

Drop the entire collection (cleanup). Confirms all data is removed.

In [ ]:
resp = requests.delete(f"{BASE_URL}/collections/{COLLECTION}")
print(f"── DELETE /collections/{COLLECTION} ──")
print(f"HTTP {resp.status_code}")

# Verify collection is gone
resp2 = requests.get(f"{BASE_URL}/collections")
collections = resp2.json()
names = [c.get("name") for c in (collections if isinstance(collections, list) else [])]
assert COLLECTION not in names, f"{COLLECTION} should be deleted!"
print(f"✅ Collection '{COLLECTION}' successfully dropped")
print(f"Remaining collections: {names}")